In [1]:
using Plots
using Distributions, Random, Statistics
using LinearAlgebra, SparseArrays, JSON
using Colors
using CSV
using DataFrames
using MLJ
using Dates
using JuMP, Ipopt
using LinearAlgebra

purple = RGB(122/256, 30/256, 71/256)
blue   = RGB(0/256, 39/256, 76/256)
mygreen = RGB(83/256, 165/256, 132/256)

In [ ]:
# Load all four locations
dfs = [CSV.read("location$i.csv", DataFrame; types = Dict(:Time => String)) for i in 1:4]

# Convert timestamps and assign location IDs
for (i, data) in enumerate(dfs)
    data.Time = DateTime.(data.Time, dateformat"yyyy-mm-dd HH:MM:SS")
    data.LocationID = fill(i, nrow(data))
end

# Combine all locations and sort by location and time
df = vcat(dfs...)
sort!(df, [:LocationID, :Time])

m     = nrow(df)
times = df.Time
t     = Float64.(df.Power)

# Normalize weather features across all four locations
x10m  = Float64.(df.windspeed_10m) ./ maximum(df.windspeed_10m)
x100m = Float64.(df.windspeed_100m) ./ maximum(df.windspeed_100m)
x10g  = Float64.(df.windgusts_10m) ./ maximum(df.windgusts_10m)

# 24-hour groups
hours_per_group = 24
num_groups = div(m, hours_per_group)

m_used = num_groups * hours_per_group

times = times[1:m_used]
t     = t[1:m_used]
x10m  = x10m[1:m_used]
x100m = x100m[1:m_used]
x10g  = x10g[1:m_used]

location_ids = df.LocationID[1:m_used]

train, val, test = partition(1:num_groups, 0.8, 0.1;shuffle = false,rng = 120)

c = collect(0.0:0.03:1)
num_centers = length(c)

function rbf_features(xs, γ)

    Φ = zeros(length(xs), num_centers)
    for i in eachindex(xs), j in 1:num_centers
        Φ[i, j] = exp(-(γ * abs(xs[i] - c[j]))^2)
    end
    
    return Φ
end

weather_design_matrix(x1, x2, x3, γ) = hcat(rbf_features(x1, γ), rbf_features(x2, γ), rbf_features(x3, γ))

grouped_design_matrix(x1, x2, x3, lag_power, γ) = hcat(weather_design_matrix(x1, x2, x3, γ), lag_power)
Y_all = t

group_to_indices(groups, hours_per_group) = vcat([((g - 1) * hours_per_group + 1):(g * hours_per_group) for g in groups]...)

train_idx = group_to_indices(train, hours_per_group)
val_idx   = group_to_indices(val, hours_per_group)
test_idx  = group_to_indices(test, hours_per_group)

train_idx_fit = filter(i -> i > 1, train_idx)

function train_rbf(Φ, t;α = 0.01,n_epochs = 20,λ = 0.001,reg_type = :l2,seed = 128)

    rng = MersenneTwister(seed)
    ntrain = size(Φ, 1)
    nfeat  = size(Φ, 2)
    w_0 = 0.0
    w   = zeros(nfeat)

    for epoch in 1:n_epochs
        for i in shuffle(rng, 1:ntrain)

            ϕi = @view Φ[i, :]
            err = (w_0 + dot(ϕi, w)) - t[i]
            w_0 -= α * err
            w   .-= α * err .* ϕi

            if reg_type == :l2
                w .-= α * λ .* w

            elseif reg_type == :l1
                w .= sign.(w) .* max.(0, abs.(w) .- α * λ)

            end
        end
    end

    return w_0, w
end

predict_rbf(Φ, w_0, w) = w_0 .+ Φ * w

rmse(y_pred, y_actual) = sqrt(mean((y_pred .- y_actual).^2))


# ---------- Recursive prediction ----------

function predict_recursive(Φ_weather, groups, t, times, location_ids, w_0, w)

    predictions = Float64[]

    for group in groups

        # Initialize with actual power from the previous day
        first_idx = first(group)
        previous_idx = first_idx - 24

        if previous_idx < 1 ||
           location_ids[previous_idx] != location_ids[first_idx] ||
           times[first_idx] - times[previous_idx] != Hour(24)

            error("Previous-day observation unavailable for group $group")
        end

        previous_power = t[previous_idx]

        for idx in group

            # Current weather features
            ϕ_weather = @view Φ_weather[idx, :]

            # Append previous power
            ϕ = vcat(collect(ϕ_weather), previous_power)

            # Predict current power
            prediction = w_0 + dot(ϕ, w)
            prediction = clamp(prediction, 0.0, 1.0)

            push!(predictions, prediction)

            # Use prediction for the next hour
            previous_power = prediction

        end
    end

    return predictions
end


# ---------- Hyperparameter search ----------

λ_grid = [1e-2, 2e-2, 3e-2]
γ_grid = [3, 3.25, 3.5, 3.75, 4.5]

best = (rmse = Inf, λ = NaN, γ = NaN)

for γ in γ_grid, λ in λ_grid

    Φ_weather = weather_design_matrix(x10m, x100m, x10g, γ)
    lag_train = t[train_idx_fit .- 1]
    
    Φ_train = grouped_design_matrix(x10m[train_idx_fit],x100m[train_idx_fit],x10g[train_idx_fit],lag_train,γ)
    Y_train = Y_all[train_idx_fit]

    w_0, w = train_rbf(Φ_train, Y_train; λ = λ)

    Y_val_pred = predict_recursive(Φ_weather, groups[val], t, times, location_ids, w_0, w)
    Y_val = Y_all[val_idx]

    score = rmse(Y_val_pred, Y_val)

    if isfinite(score) && score < best.rmse
        global best = (rmse = score,λ = λ,γ = γ)
    end
end

println("Best: ", best)

# ---------- Final fit and test evaluation ----------

γ, λ = best.γ, best.λ

Φ_weather = weather_design_matrix(x10m, x100m, x10g, γ)

# Training lag uses actual previous-hour measurements
lag_train = t[train_idx_fit .- 1]

Φ_train = grouped_design_matrix(x10m[train_idx_fit],x100m[train_idx_fit],x10g[train_idx_fit],lag_train,γ)
Y_train = Y_all[train_idx_fit]

# Train final model
w_0, w = train_rbf(Φ_train, Y_train; λ = λ)

# ---------- Validation predictions ----------

Y_val_pred = predict_recursive(Φ_weather,val_idx,initial_val_lag,w_0,w)
Y_val = Y_all[val_idx]
val_rmse = rmse(Y_val_pred, Y_val)

println("Final validation RMSE = ", round(val_rmse, digits = 4))


# ---------- Test predictions ----------

Y_test_pred = predict_recursive(
    Φ_weather, groups[test], t, times, location_ids, w_0, w
)
Y_test = Y_all[test_idx]
test_rmse = rmse(Y_test_pred, Y_test)

println("Test RMSE = ", round(test_rmse, digits = 4))


# ---------- Predictions for every hourly observation ----------

Y_pred = fill(NaN, m_used)

# Training predictions use actual previous-hour measurements
Φ_train_all = grouped_design_matrix(x10m[train_idx_fit],x100m[train_idx_fit],x10g[train_idx_fit],t[train_idx_fit .- 1],γ)

Y_pred[train_idx_fit] .= clamp.(predict_rbf(Φ_train_all, w_0, w),0.0,1.0)
# Validation predictions
Y_pred[val_idx] .= Y_val_pred
# Test predictions
Y_pred[test_idx] .= Y_test_pred

# ---------- Plot each 24-hour group ----------

for group in test[1:min(10, length(test))]

    idx = ((group - 1) * hours_per_group + 1):(group * hours_per_group)

    plo = scatter(times[idx], Y_all[idx], c = blue, label = "Actual power", ms = 3, markerstrokewidth = 0, frame = :box, xlabel = "time", ylabel = "power output [p.u.]", title = "24-hour group $group (Test)")
    plot!(plo, times[idx], Y_pred[idx], lw = 1.5, c = purple, label = "RBF prediction")
    plot!(plo, ylims = (-0.25, 1.2))

    display(plo)
end


UndefVarError: UndefVarError: `groups` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [ ]:

# ---------- Load all four locations ----------

dfs = [CSV.read("location$i.csv", DataFrame; types = Dict(:Time => String)) for i in 1:4]

# Convert timestamps and assign location IDs
for (i, data) in enumerate(dfs)
    data.Time = DateTime.(data.Time, dateformat"yyyy-mm-dd HH:MM:SS")
    data.LocationID = fill(i, nrow(data))
end

# Combine all locations and sort by location and time
df = vcat(dfs...)
sort!(df, [:LocationID, :Time])

m     = nrow(df)
times = df.Time
t     = Float64.(df.Power)

location_ids = df.LocationID

# ---------- Normalize weather features ----------

x10m  = Float64.(df.windspeed_10m) ./ maximum(df.windspeed_10m)
x100m = Float64.(df.windspeed_100m) ./ maximum(df.windspeed_100m)
x10g  = Float64.(df.windgusts_10m) ./ maximum(df.windgusts_10m)

# ---------- 24-hour groups, separated by location ----------

hours_per_group = 24

groups = Vector{Int}[]

for loc in unique(location_ids)

    loc_idx = findall(==(loc), location_ids)

    for i in 1:hours_per_group:(length(loc_idx) - hours_per_group + 1)

        push!(groups, loc_idx[i:i + hours_per_group - 1])

    end
end

num_groups = length(groups)

# Partition groups chronologically
train, val, test = partition(1:num_groups, 0.8, 0.1;shuffle = false,rng = 120)

# Convert group selections to observation indices
train_idx = vcat(groups[train]...)
val_idx   = vcat(groups[val]...)
test_idx  = vcat(groups[test]...)

# Exclude observations without a valid previous-hour measurement
train_idx_fit = filter(i -> i > 1 && location_ids[i] == location_ids[i-1] &&times[i] - times[i-1] == Hour(1),train_idx)

# ---------- RBF features ----------

c = collect(0.0:0.01:1)
num_centers = length(c)

function rbf_features(xs, γ)

    Φ = zeros(length(xs), num_centers)

    for i in eachindex(xs), j in 1:num_centers
        Φ[i, j] = exp(-(γ * abs(xs[i] - c[j]))^2)
    end

    return Φ
end

weather_design_matrix(x1, x2, x3, γ) = hcat(rbf_features(x1, γ),rbf_features(x2, γ),rbf_features(x3, γ))
grouped_design_matrix(x1, x2, x3, lag_power, γ) = hcat(weather_design_matrix(x1, x2, x3, γ),lag_power)

Y_all = t

# ---------- RBF training ----------

function train_rbf(Φ, t; α = 0.01, n_epochs = 15, λ = 0.001,reg_type = :l2, seed = 128)

    rng = MersenneTwister(seed)

    ntrain = size(Φ, 1)
    nfeat  = size(Φ, 2)

    w_0 = 0.0
    w   = zeros(nfeat)

    for epoch in 1:n_epochs

        for i in shuffle(rng, 1:ntrain)

            ϕi = @view Φ[i, :]

            err = (w_0 + dot(ϕi, w)) - t[i]

            w_0 -= α * err
            w   .-= α * err .* ϕi

            if reg_type == :l2
                w .-= α * λ .* w

            elseif reg_type == :l1
                w .= sign.(w) .* max.(0, abs.(w) .- α * λ)

            end
        end
    end

    return w_0, w
end

predict_rbf(Φ, w_0, w) = w_0 .+ Φ * w

rmse(y_pred, y_actual) = sqrt(mean((y_pred .- y_actual).^2))

# ---------- Recursive prediction for validation and testing ----------

function predict_recursive(Φ_weather, groups, t, times, location_ids, w_0, w)

    predictions = Float64[]

    for group in groups

        # Initialize with actual power from the last hour of the previous day
        first_idx = first(group)
        previous_time = times[first_idx] - Hour(1)

        previous_idx = findfirst(
            i -> location_ids[i] == location_ids[first_idx] &&
                 times[i] == previous_time,
            eachindex(times)
        )

        if previous_idx === nothing
            error("Previous-hour observation unavailable for group $group")
        end

        previous_power = t[previous_idx]

        for idx in group
            # Current weather features
            ϕ_weather = @view Φ_weather[idx, :]

            # Append previous power
            ϕ = vcat(collect(ϕ_weather), previous_power)

            # Make current prediction
            prediction = w_0 + dot(ϕ, w)

            # Keep prediction within normalized power range
            prediction = clamp(prediction, 0.0, 1.0)
            push!(predictions, prediction)

            # Use prediction for the next hour
            previous_power = prediction

        end
    end

    return predictions
end

# ---------- Hyperparameter search ----------
λ_grid = [1e-2, 2e-2, 5e-2]
γ_grid = [ 7, 8, 10, 25]

best = (rmse = Inf, λ = NaN, γ = NaN)

for γ in γ_grid, λ in λ_grid

    # Weather features
    Φ_weather = weather_design_matrix(x10m, x100m, x10g, γ)
    # Training uses actual previous-hour power
    lag_train = t[train_idx_fit .- 1]
    Φ_train = grouped_design_matrix(x10m[train_idx_fit],x100m[train_idx_fit],x10g[train_idx_fit],lag_train,γ)
    Y_train = Y_all[train_idx_fit]

    # Fit model
    w_0, w = train_rbf(Φ_train, Y_train; λ = λ)

    # Validation uses previous-day actual power for initialization
    # and recursive predictions for subsequent hours
    Y_val_pred = predict_recursive(Φ_weather, groups[val], t, times, location_ids, w_0, w)
    Y_val = Y_all[val_idx]

    score = rmse(Y_val_pred, Y_val)

    if isfinite(score) && score < best.rmse
        global best = (rmse = score,λ = λ,γ = γ)
    end
end

println("Best: ", best)

# ---------- Final fit and evaluation ----------
γ, λ = best.γ, best.λ
Φ_weather = weather_design_matrix(x10m, x100m, x10g, γ)

# Training lag uses actual previous-hour measurements
lag_train = t[train_idx_fit .- 1]
Φ_train = grouped_design_matrix(x10m[train_idx_fit],x100m[train_idx_fit],x10g[train_idx_fit],lag_train,γ)
Y_train = Y_all[train_idx_fit]

# Train final model
w_0, w = train_rbf(Φ_train, Y_train; λ = λ)

# ---------- Validation predictions ----------
Y_val_pred = predict_recursive(Φ_weather, groups[val], t, times, location_ids, w_0, w)
Y_val = Y_all[val_idx]
val_rmse = rmse(Y_val_pred, Y_val)

println("Final validation RMSE = ", round(val_rmse, digits = 4))

# ---------- Test predictions ----------
Y_test_pred = predict_recursive(Φ_weather, groups[test], t, times, location_ids, w_0, w)
Y_test = Y_all[test_idx]
test_rmse = rmse(Y_test_pred, Y_test)

println("Test RMSE = ", round(test_rmse, digits = 4))

# ---------- Predictions for every hourly observation ----------
Y_pred = fill(NaN, m)

# Training predictions use actual previous-hour measurements
Φ_train_all = grouped_design_matrix(x10m[train_idx_fit],x100m[train_idx_fit],x10g[train_idx_fit],t[train_idx_fit .- 1],γ)
Y_pred[train_idx_fit] .= clamp.(predict_rbf(Φ_train_all, w_0, w),0.0,1.0)

# Validation predictions
Y_pred[val_idx] .= Y_val_pred
# Test predictions
Y_pred[test_idx] .= Y_test_pred

# ---------- Plot each 24-hour test group ----------
for group_num in test[1:min(10, length(test))]

    idx = groups[group_num]

    plo = scatter(times[idx],Y_all[idx],c = blue,label = "Actual power",ms = 3,markerstrokewidth = 0,frame = :box,xlabel = "time",ylabel = "power output [p.u.]",title = "24-hour group $group_num (Test)")
    plot!(plo,times[idx],Y_pred[idx],lw = 1.5,c = purple,label = "RBF prediction")
    plot!(plo, ylims = (-0.25, 1.2))

    display(plo)
end
